### Ingestión del archivo "movie.csv"

In [0]:
%run "../includes/configuration"





In [0]:
%run "../includes/common_functions"






In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")
















### Paso 1 - Leer el archivo CSV usando "DataFrameReader" de Spark
### La documentación para los comandos de Spark la sacamos de https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html

In [0]:
#movie_df = spark.read.csv("abfss://bronze@moviee.dfs.core.windows.net/movie.csv")

movie_df = spark.read.csv(f"{bronze_folder_path}/movie.csv")












In [0]:
movie_df.show()

In [0]:
display(movie_df)

In [0]:
#movie_df = spark.read.option("header",True).csv("abfss://bronze@moviee.dfs.core.windows.net/movie.csv")

movie_df = spark.read.option("header",True).csv(f"{bronze_folder_path}/movie.csv")






In [0]:
display(movie_df)

In [0]:
movie_df.printSchema()

In [0]:
display(movie_df.describe())

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, DateType








In [0]:
movie_schema = StructType(fields = [
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("budget", DoubleType(), True),
    StructField("homePage", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("yearReleaseDate", IntegerType(), True),
    StructField("ReleaseDate", DateType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("durationTime", IntegerType(), True),
    StructField("movieStatus", StringType(), True),
    StructField("tagline", StringType(), True),
    StructField("voteAverage", DoubleType(), True),
    StructField("voteCount", IntegerType(), True)
])

In [0]:
#movie_df = spark.read.option("header",True).schema(movie_schema).csv("abfss://bronze@moviee.dfs.core.windows.net/movie.csv")

movie_df = spark.read.option("header",True).schema(movie_schema).csv(f"{bronze_folder_path}/movie.csv")






In [0]:
movie_df.printSchema()

### Paso 2 - Seleccionar solo las columnas requeridas

In [0]:
movies_selected_df = movie_df.select("movieId","title","budget","popularity","yearReleaseDate","ReleaseDate","revenue","durationTime","voteAverage","voteCount")






In [0]:
movies_selected_df = movie_df.select(movie_df.movieId,movie_df.title,movie_df.budget,movie_df.popularity,movie_df.yearReleaseDate,movie_df.ReleaseDate,movie_df.revenue,movie_df.durationTime,movie_df.voteAverage,movie_df.voteCount)


In [0]:
movies_selected_df = movie_df.select(movie_df["movieId"],movie_df["title"],movie_df["budget"],movie_df["popularity"],movie_df["yearReleaseDate"],movie_df["ReleaseDate"],movie_df["revenue"],movie_df["durationTime"],movie_df["voteAverage"],movie_df["voteCount"])


In [0]:
from pyspark.sql.functions import col






In [0]:
movies_selected_df = movie_df.select(col("movieId"),col("title"),col("budget"),col("popularity"),col("yearReleaseDate"),col("ReleaseDate"),col("revenue"),col("durationTime"),col
                                     
                                     
                                     
                                     
                                     
                                     
                                     ("voteAverage"),col("voteCount"))

In [0]:
display(movies_selected_df)

### Paso 3 - Cambiar el nombre de las columnas según lo requerido

In [0]:
movies_renamed_of = movies_selected_df \
                        .withColumnRenamed("movieId", "movie_id") \
                        .withColumnRenamed("yearReleaseDate", "year_release_date") \
                        .withColumnRenamed("ReleaseDate", "release_date") \
                        .withColumnRenamed("durationTime", "duration_time") \
                        .withColumnRenamed("voteAverage", "vote_average") \
                        .withColumnRenamed("voteCount", "vote_count") 



                        



In [0]:
display(movies_renamed_of)

### Paso 4 - Agregar la columnas "ingestion_date" y "environment" al DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit











In [0]:
#movies_final_df = movies_renamed_of.withColumn("ingestion_date", current_timestamp()) \
#                                    .withColumn("environment", lit("production"))


movies_final_df = add_ingestion_date(movies_renamed_of) \
                    .withColumn("environment", lit(v_environment))


















                                    

In [0]:
movies_final_df = movies_renamed_of.withColumns({"ingestion_date": current_timestamp(), "environment": lit("production")})



In [0]:
display(movies_final_df)

### Paso 5 - Escribir datos en el datalake en formato "Parquet"

In [0]:
#movies_final_df.write.mode("overwrite").partitionBy("year_release_date").parquet("abfss://silver@moviee.dfs.core.windows.net/movies")

movies_final_df.write.mode("overwrite").partitionBy("year_release_date").parquet(f"{silver_folder_path}/movies")





































In [0]:
#df = spark.read.parquet("abfss://silver@moviee.dfs.core.windows.net/movies")


df = spark.read.parquet(f"{silver_folder_path}/movies")


In [0]:
display(df)

### Paso 6 - Escribir datos en una managed table en el contenedor silver

#### Esto es un cambio posterior en el notebook, pasaría a reemplazar al paso 5, ya que ahora no quiero más generar archivos de salida en la capa silver a partir del dataframe, sino meter esa info en una tabla administrada por spark, pero dejo igual lo del paso 5, ya que esto se crea dentro de la carpeta _unitystorage y no pisa lo del paso 5

In [0]:
movies_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.movies")







In [0]:
%sql
SELECT * FROM movie_silver.movies

In [0]:
dbutils.notebook.exit("Success")






